In [ ]:
import os
import re
import glob
import tqdm
import warnings
import subprocess
import pandas as pd
warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None

# COMPLETE

In [2]:
# directory = "JetsonNano"
# directory = "NvidiaA100"
# paths = glob.glob(f"{directory}/synthetic-data_logger*/*")
# for path in tqdm.tqdm(paths):
#   approach = re.findall(r"(\w+)$", path).pop()
#   qps = int(re.findall(r"agg(\d+)", path).pop())
#   targets = glob.glob(f"{directory}/synthetic_running*x/qps{qps}")
#   dest = None
#   for target in targets:
#     if os.path.exists(f"{target}/{approach}"):
#       continue
#     dest = target
#     break
#   if not dest:
#     running = len(targets) + 1
#     dest = f"{directory}/synthetic_running{running}x/qps{qps}"
#   os.makedirs(dest, exist_ok=True)
#   bash_command = f"cp -r {path} {dest}"
#   subprocess.run(bash_command, shell=True, check=True)

# CONCAT

In [ ]:
directory = "JetsonNano"
directory = "NvidiaA100"
paths = sorted(glob.glob(f"{directory}/synthetic_running*x"))
N = len(paths)
M = 0
for path in paths:
  approaches = { os.path.basename(el) for el in glob.glob(f"{directory}/synthetic_running*x/qps*/*") }
  M = max(M, len(approaches))
for i in tqdm.tqdm(range(1, N + 1)):
  subdirs = glob.glob(f"{directory}/synthetic_running{i}x/qps*")
  for subdir in subdirs:
    experiments = glob.glob(f"{subdir}/*")
    for approach in approaches:
      path = f"{subdir}/{approach}"
      if os.path.exists(path):
        continue
      for j in range(N, i, -1):
        src = path.replace(f"running{i}", f"running{j}")
        if not os.path.exists(src):
          continue
        dest = os.path.dirname(path)
        os.makedirs(dest, exist_ok=True)
        bash_command = f"cp -r {src} {dest}"
        subprocess.run(bash_command, shell=True, check=True)
        break